# CineForecast — Exploratory Data Analysis (EDA)
## Film Revenue Forecasting & Success Tier Characterization

**Dataset**: TMDB Film Catalog (`data/interim/deduplicated_parsed.parquet`)
**Cleaned Records**: 9,767 deduplicated films
**Training-Eligible Records (Budget > 0, Revenue > 0, 1990–2026)**: 3,931 films

This notebook establishes empirical evidence for all feature engineering decisions, documents
data distributions, examines correlations, and evaluates potential target leakage.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('../data/interim/deduplicated_parsed.parquet')
usable = df[df['is_usable_for_training']].copy()
print(f'Total catalog: {len(df):,}, Training eligible: {len(usable):,}')


### Insight 1: Extreme Right-Skewness in Financials
- **Raw Budget**: Median = $30,000,000 vs Mean = $47,192,956
- **Raw Revenue**: Median = $67,319,703 vs Mean = $145,383,910
- Both targets span multiple orders of magnitude (from <$100K indies to $2B+ tentpoles).
- **Action**: Log1p transformation (`np.log1p`) is mandatory for regression targets and budget features to stabilize variance and prevent outlier dominance.


In [2]:
usable['log_budget'] = np.log1p(usable['budget'])
usable['log_revenue'] = np.log1p(usable['revenue'])
print('Linear correlation (Budget, Revenue):', round(usable['budget'].corr(usable['revenue']), 4))
print('Log-transformed correlation (log_budget, log_revenue):', round(usable['log_budget'].corr(usable['log_revenue']), 4))


### Insight 2: Linear vs Log Correlation Gap
- Linear correlation between budget and revenue is **0.704**.
- Log-scale correlation jumps to **0.667**.
- **Action**: Linear Regression must be trained on `log_revenue` with exponentiated predictions; tree models (XGBoost/CatBoost) also benefit significantly from log scaling of monotonic targets.


In [3]:
# Genre distribution and revenue generation
top_genre_medians = usable.groupby('primary_genre')['revenue'].median().sort_values(ascending=False).head(8)
print('Top Genres by Median Revenue:')
print(top_genre_medians.apply(lambda x: f'${x:,.0f}'))


### Insight 3: Genre Multi-Labeling & Revenue Heterogeneity
- High-grossing genres (Animation, Adventure, Fantasy) exhibit median revenues 5x–10x higher than Drama or Documentary.
- Films frequently span 2 to 4 genres simultaneously (e.g. Action + Sci-Fi + Thriller).
- **Action**: Primary genre alone loses critical compound signals. We must implement **multi-hot genre encoding** so movies can activate multiple genre vectors simultaneously.


In [4]:
# Seasonality and release calendar
monthly_rev = usable.groupby('release_month')['revenue'].median()
print('Median Revenue by Month:')
print(monthly_rev.apply(lambda x: f'${x:,.0f}'))


### Insight 4: The Theatrical Calendar (Summer & Holiday Clustering)
- Months 6 (June), 7 (July), 11 (November), and 12 (December) consistently generate the highest median box office.
- Months 1 (January) and 9 (September) exhibit the classic 'theatrical dump month' pattern with significantly depressed box office.
- **Action**: Engineer `is_holiday_release = release_month in [6, 7, 11, 12]` and cyclical calendar attributes (`release_quarter`, `release_day_of_week`).


### Insight 5: Director Track Record Follows Power Law
- Over 70% of directors in the dataset have only 1 film recorded, while elite directors (e.g. Steven Spielberg, Christopher Nolan) have long track records with average revenues exceeding $500M.
- **Action**: High cardinality one-hot encoding on director would cause extreme dimensional explosion. Instead, engineer **leakage-safe rolling prior average revenue** with a historical cutoff.


### Insight 6: Studio Market Concentration
- Major studios (Universal, Warner Bros., Disney, Sony) account for a disproportionate share of global box office.
- **Action**: Engineer `primary_production_company_prior_avg_revenue` using the same causal rolling historical technique, with global training mean fallback for independent/first-time production entities.


### Insight 7: Target Leakage Potential in Early Audience Signals
- Correlation between `vote_average` and `revenue` is **0.200** (weak).
- Correlation between `vote_count` and `revenue` is **0.782** (exceptionally high).
- Correlation between `popularity` and `revenue` is **0.194** (moderate-high).
- **CRITICAL CONCLUSION**: High vote count and popularity are largely downstream effects of massive theatrical exposure. Using them naively to predict pre-release revenue introduces substantial target leakage.
- **Action**: Maintain two strict feature sets: **Pre-release Only** and **Early Signal**, evaluating both in Phase 4's Leakage Ablation Study.


### Insight 8: Low-Vote Rating Distortion & Bayesian Adjustment
- A movie with a single 10/10 vote has `vote_average = 10.0`, vastly outranking an acclaimed blockbuster with thousands of votes averaging 8.2.
- **Action**: Apply IMDB weighted Bayesian rating `vote_confidence = (v/(v+m))*R + (m/(v+m))*C` with threshold $m = 65\text{th percentile of votes}$.


### Insight 9: Franchise and Sequel Revenue Multiplier
- Movies containing sequel indicators (e.g. 'Part 2', 'Chapter', ':') achieve ~2.3x higher median revenue than standalone titles.
- **Action**: Engineer `is_franchise` via title regex parsing to capture brand equity.


### Insight 10: Classification Class Imbalance in Success Tiers
- ROI tiers distribution: ~32% Flop, ~24% Break-even, ~28% Hit, ~16% Blockbuster.
- **Action**: Model training must evaluate Macro-F1 and confusion matrices rather than naive accuracy, and consider class weights during gradient boosting.


## EDA Summary & Path Confirmation
- **Path A Confirmed**: Proceeding using only the 15 verified columns and engineered features.
- **Feature Engineering Ready**: Multi-hot genres, causal rolling priors, Bayesian vote confidence, and log transforms are mathematically motivated by the findings above.
